# BirdCLEF 2026 — Inference v21
## ResNet18 (v20) + EfficientNet-B0 (v20) + PerchHead (v21, domain-fixed)
### 15 models total: 3 archs × 5 folds

**What changed vs v20:**
- `perch_head` weights retrained on in-domain soundscape embeddings (v21)
- Fixes train/inference domain mismatch that caused v20 regression

**Required Kaggle input datasets:**
- `birdclef-2026` (competition data)
- `chiragggg/birdclef-2026-v20-model` (mel model weights: resnet18_v20 + efficientnet_b0_v20)
- `chiragggg/birdclef-2026-perch-weights-v21` (perch_head_v21_fold*.pt)
- `google/bird-vocalization-classifier` (Perch 2.0 TF model, TF2 > perch_v2 > v2)

In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

# Hide GPU from TensorFlow (Perch runs CPU-only at inference)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
try:
    import tensorflow as tf
    _TF_OK = True
except Exception:
    _TF_OK = False

import torch
import torch.nn as nn
import timm
from torch.cuda.amp import autocast
from tqdm import tqdm
import gc

warnings.filterwarnings("ignore")

CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,
    mel_archs     = ["resnet18", "efficientnet_b0"],   # v20 weights
    folds         = 5,
    tta_offsets   = [-1.25, 0.0, 1.25],
    device        = "cuda" if torch.cuda.is_available() else "cpu",
    perch_sr      = 32000,
    perch_emb_dim = 1536,
)

device = torch.device(CFG["device"])
torch.set_num_threads(os.cpu_count() or 4)
n_mel_models   = len(CFG["mel_archs"]) * CFG["folds"]
n_perch_models = CFG["folds"]
print("v21 inference config ready")
print(f"   Device          : {device}")
print(f"   TF OK           : {_TF_OK}")
print(f"   Mel models (v20): {n_mel_models}  ({CFG['mel_archs']} x {CFG['folds']} folds)")
print(f"   Perch models    : {n_perch_models}  (perch_head_v21 x {CFG['folds']} folds)")
print(f"   TTA crops       : {len(CFG['tta_offsets'])}")

In [ ]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
TEST_AUDIO = _first_existing(
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
)
SAMPLE_SUB = _first_existing(
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
)

# Mel model weights (resnet18_v20 + efficientnet_b0_v20)
MEL_CKPT_DIR = _first_existing(
    "/kaggle/input/birdclef-2026-v20-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v20-model",
    "/kaggle/working",
)

# Perch head v21 weights
PERCH_CKPT_DIR = _first_existing(
    "/kaggle/input/birdclef-2026-perch-weights-v21",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v21",
    "/kaggle/working",
)

# Perch 2.0 TF model
PERCH_MODEL_PATH = Path(_first_existing(
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/bird-vocalization-classifier",
))

# Working dir for test embeddings cache
EMBD_DIR = Path("/kaggle/working/test_embs")
EMBD_DIR.mkdir(parents=True, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f"✅ {n_classes} species from taxonomy.csv")
print(f"   TEST_AUDIO      : {TEST_AUDIO}")
print(f"   MEL_CKPT_DIR    : {MEL_CKPT_DIR}")
print(f"   PERCH_CKPT_DIR  : {PERCH_CKPT_DIR}")
print(f"   PERCH_MODEL_PATH: {PERCH_MODEL_PATH}  (exists={PERCH_MODEL_PATH.exists()})")


In [ ]:
# === CELL 3: MEL HELPER ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ logmel_from_wave defined")

In [ ]:
# === CELL 4: MODEL ARCHITECTURES ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch
        if arch == "resnet18":
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features
        elif arch == "resnet50":
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features
        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features
        else:
            raise ValueError(f"Unsupported arch: {arch}")

        # v20 head: Linear(in→512) → ReLU → Dropout(0.4) → Linear(512→n_classes)
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


class PerchHead(nn.Module):
    """MLP head that runs on top of Perch 2.0 1536-dim embeddings."""
    def __init__(self, n_classes: int, emb_dim: int = 1536):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


print("✅ BirdCLEFModel and PerchHead defined")


In [ ]:
# === CELL 5: LOAD CHECKPOINTS ===
mel_models  = []   # resnet18_v20 + efficientnet_b0_v20
emb_models  = []   # perch_head_v21
missing     = []

# Load mel models (v20 weights)
for arch in CFG["mel_archs"]:
    for fold_idx in range(CFG["folds"]):
        ckpt_path = Path(MEL_CKPT_DIR) / f"{arch}_v20_fold{fold_idx}.pt"
        if not ckpt_path.exists():
            missing.append(str(ckpt_path))
            continue
        m = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
        m.eval()
        mel_models.append(m)
        print(f"   ✅ Loaded {ckpt_path.name}")

# Load perch_head models (v21 weights)
for fold_idx in range(CFG["folds"]):
    ckpt_path = Path(PERCH_CKPT_DIR) / f"perch_head_v21_fold{fold_idx}.pt"
    if not ckpt_path.exists():
        missing.append(str(ckpt_path))
        continue
    m = PerchHead(n_classes=n_classes, emb_dim=CFG["perch_emb_dim"]).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    m.eval()
    emb_models.append(m)
    print(f"   ✅ Loaded {ckpt_path.name}")

if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) not found:")
    for p in missing:
        print(f"      {p}")

total_models = len(mel_models) + len(emb_models)
print(f"\n✅ {len(mel_models)} mel models + {len(emb_models)} perch models = {total_models} total")


In [ ]:
# === CELL 6: PERCH 2.0 TEST EMBEDDING PRECOMPUTE (TF, CPU-only) ===
# Extracts 1536-dim embeddings for every 5s window using SAME windowing as training.
# Caches to /kaggle/working/test_embs/ — skips existing files on re-run.
# TF model is freed after this cell.

_PERCH_BATCH_INF = 16
_perch_ready_inf = False

if not _TF_OK:
    print("TensorFlow not available — perch_head will output neutral 0.5")
elif not PERCH_MODEL_PATH.exists():
    print(f"WARNING: Perch model not found at {PERCH_MODEL_PATH}")
    print("Add 'google/bird-vocalization-classifier' (TF2 > perch_v2 > v2) to notebook inputs.")
else:
    try:
        _sample_sub_df = pd.read_csv(SAMPLE_SUB)
        _total_rows    = len(_sample_sub_df)
        _existing_embs = len(list(EMBD_DIR.glob("*.npy")))

        if _existing_embs >= _total_rows:
            print(f"Reusing {_existing_embs} cached test embeddings")
            _perch_ready_inf = True
        else:
            print(f"Loading Perch 2.0 from {PERCH_MODEL_PATH} ...")
            tf.config.set_visible_devices([], "GPU")
            _perch_model  = tf.saved_model.load(str(PERCH_MODEL_PATH))
            _perch_infer  = _perch_model.signatures["serving_default"]
            _perch_target = CFG["perch_sr"] * CFG["seconds"]   # 160,000 samples

            _audio_batch, _name_batch = [], []

            def _flush_batch_inf():
                try:
                    _audio_tf = tf.constant(np.stack(_audio_batch), dtype=tf.float32)
                    _out  = _perch_infer(inputs=_audio_tf)
                    _embs = _out["embedding"].numpy()
                    for _n, _e in zip(_name_batch, _embs):
                        np.save(str(EMBD_DIR / (_n + ".npy")), _e.astype(np.float32))
                except Exception as _fe:
                    print(f"  WARNING: batch flush failed: {_fe}")
                finally:
                    _audio_batch.clear()
                    _name_batch.clear()

            _rows_by_sc = defaultdict(list)
            for _rid in _sample_sub_df["row_id"]:
                _sc, _es = _rid.rsplit("_", 1)
                _rows_by_sc[_sc].append(int(_es))

            _failed = 0
            for _sc_id, _end_secs in tqdm(_rows_by_sc.items(), desc="Perch embed (test)"):
                _apath = Path(TEST_AUDIO) / f"{_sc_id}.ogg"
                if not _apath.exists():
                    for _ext in (".wav", ".flac"):
                        _fb = _apath.with_suffix(_ext)
                        if _fb.exists():
                            _apath = _fb
                            break
                    else:
                        _failed += 1
                        continue
                try:
                    _y, _sr0 = sf.read(str(_apath), always_2d=False)
                    if _y.ndim == 2:
                        _y = _y.mean(axis=1)
                    if _sr0 != CFG["perch_sr"]:
                        _y = librosa.resample(
                            _y.astype(np.float32), orig_sr=_sr0, target_sr=CFG["perch_sr"]
                        )
                    _y = _y.astype(np.float32)
                    for _es in _end_secs:
                        _emb_name = f"{_sc_id}_{_es}"
                        if (EMBD_DIR / (_emb_name + ".npy")).exists():
                            continue
                        _end_samp   = int(_es * CFG["perch_sr"])
                        _start_samp = max(0, _end_samp - _perch_target)
                        _clip = _y[_start_samp:_end_samp]
                        if len(_clip) < _perch_target:
                            _clip = np.pad(_clip, (0, _perch_target - len(_clip)))
                        _audio_batch.append(_clip)
                        _name_batch.append(_emb_name)
                        if len(_audio_batch) >= _PERCH_BATCH_INF:
                            _flush_batch_inf()
                except Exception as _exc:
                    print(f"  WARNING: failed {_apath.name}: {_exc}")
                    _failed += 1

            if _audio_batch:
                _flush_batch_inf()

            try:
                del _perch_model, _perch_infer
                tf.keras.backend.clear_session()
                gc.collect()
            except Exception:
                pass

            _saved = len(list(EMBD_DIR.glob("*.npy")))
            print(f"Test Perch embeddings: {_saved} saved  ({_failed} soundscapes failed)")
            print("TF model freed — proceeding to PyTorch inference")
            _perch_ready_inf = _saved > 0

    except Exception as _cell_exc:
        print(f"WARNING: Perch precompute failed: {_cell_exc}")
        print("perch_head will output neutral 0.5")
        _perch_ready_inf = False

In [ ]:
# === CELL 7: PREDICT FUNCTION ===
_use_amp    = (device.type == "cuda")
win_samples = CFG["sr"] * CFG["seconds"]
win_frames  = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1
_n_tta      = len(CFG["tta_offsets"])
_pad_samps  = win_samples + int(max(abs(o) for o in CFG["tta_offsets"]) * CFG["sr"]) + 1
_CHUNK      = 512
_N_WORKERS  = 2 if device.type == "cpu" else min(4, os.cpu_count() or 1)


def _extract_mel_crop(args):
    wave_padded, nom_start, offset_s = args
    off   = int(offset_s * CFG["sr"])
    start = max(0, nom_start + off)
    end_  = start + win_samples
    if end_ > len(wave_padded):
        end_  = len(wave_padded)
        start = max(0, end_ - win_samples)
    clip = wave_padded[start:end_]
    if len(clip) < win_samples:
        clip = np.pad(clip, (0, win_samples - len(clip)))
    mel = logmel_from_wave(clip, CFG["sr"])
    T = mel.shape[1]
    if T < win_frames:
        mel = np.concatenate([mel, np.zeros((mel.shape[0], win_frames - T), dtype=np.float32)], axis=1)
    elif T > win_frames:
        mel = mel[:, :win_frames]
    return mel


def predict_soundscape(audio_path: str, end_seconds, soundscape_id: str = None) -> np.ndarray:
    end_seconds = list(end_seconds)
    n_rows      = len(end_seconds)
    neutral     = np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    if not mel_models and not emb_models:
        return neutral

    all_model_probs = []

    # --- Mel-based models (resnet18 + efficientnet_b0), 3-crop TTA ---
    if mel_models:
        try:
            wave, sr0 = sf.read(audio_path, always_2d=False)
            if wave.ndim == 2:
                wave = wave.mean(axis=1)
            if sr0 != CFG["sr"]:
                wave = librosa.resample(wave, orig_sr=sr0, target_sr=CFG["sr"])
            wave = np.pad(wave.astype(np.float32), (_pad_samps, _pad_samps))

            tasks = [
                (wave, int(end_s * CFG["sr"]) + _pad_samps - win_samples, off)
                for end_s in end_seconds
                for off in CFG["tta_offsets"]
            ]
            with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                mels = list(pool.map(_extract_mel_crop, tasks))

            batch = torch.from_numpy(np.stack(mels)[:, None]).float().to(device)
            for m in mel_models:
                chunks = []
                for i in range(0, len(batch), _CHUNK):
                    with torch.inference_mode(), autocast(enabled=_use_amp):
                        p = torch.sigmoid(m(batch[i:i + _CHUNK])).float().cpu().numpy()
                    chunks.append(p)
                probs = np.concatenate(chunks, axis=0)
                probs = probs.reshape(n_rows, _n_tta, n_classes).mean(axis=1)
                all_model_probs.append(probs)
        except Exception as _e:
            print(f"  WARNING: mel inference failed for {Path(audio_path).name}: {_e}")
            all_model_probs.extend([neutral] * len(mel_models))

    # --- Perch head models (v21), no TTA — uses precomputed embeddings ---
    if emb_models:
        if _perch_ready_inf and soundscape_id is not None:
            _emb_rows = []
            for end_s in end_seconds:
                _ep = EMBD_DIR / f"{soundscape_id}_{end_s}.npy"
                _emb_rows.append(
                    np.load(str(_ep)) if _ep.exists()
                    else np.zeros(CFG["perch_emb_dim"], dtype=np.float32)
                )
            emb_batch = torch.from_numpy(np.stack(_emb_rows)).float().to(device)
            for m in emb_models:
                with torch.inference_mode(), autocast(enabled=_use_amp):
                    p = torch.sigmoid(m(emb_batch)).float().cpu().numpy()
                all_model_probs.append(p)
        else:
            all_model_probs.extend([neutral] * len(emb_models))

    if not all_model_probs:
        return neutral
    return np.mean(all_model_probs, axis=0).astype(np.float32)


print("predict_soundscape() defined")
print(f"   CPU mel workers : {_N_WORKERS}")
print(f"   TTA crops       : {_n_tta}  offsets = {CFG['tta_offsets']} s")
print(f"   Mel models      : {len(mel_models)}")
print(f"   Perch models    : {len(emb_models)}  (perch_ready={_perch_ready_inf})")

In [ ]:
# === CELL 8: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f"Sample submission rows: {len(sample_sub)}")

all_row_ids    = []
all_probs_list = []
missing_audio  = 0

sample_sub = sample_sub.copy()
sample_sub["_soundscape_id"] = sample_sub["row_id"].str.rsplit("_", n=1).str[0]

for soundscape_id, group in tqdm(
    sample_sub.groupby("_soundscape_id"),
    desc="Soundscapes",
    unit="file",
):
    audio_path = None
    for ext in [".ogg", ".wav", ".flac"]:
        cand = Path(TEST_AUDIO) / f"{soundscape_id}{ext}"
        if cand.exists():
            audio_path = str(cand)
            break

    row_ids = [str(r) for r in group["row_id"]]

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        end_seconds = [int(rid.rsplit("_", 1)[-1]) for rid in row_ids]
    except Exception as _e:
        print(f"WARNING: could not parse end_seconds: {_e}")
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        probs_all = predict_soundscape(audio_path, end_seconds, soundscape_id=soundscape_id)
    except Exception as _e:
        print(f"WARNING: predict_soundscape failed for {soundscape_id}: {_e}")
        probs_all = np.full((len(row_ids), n_classes), 0.5, dtype=np.float32)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs_all)

if missing_audio:
    print(f"⚠️  {missing_audio} soundscape(s) not found — used neutral 0.5")
print(f"\n✅ Generated {len(all_row_ids)} prediction rows")

In [ ]:
# === CELL 9: BUILD & SAVE SUBMISSION ===
probs_matrix = np.concatenate(all_probs_list, axis=0)   # (N_rows, n_classes)

sub_df = pd.DataFrame(probs_matrix, columns=species)
sub_df.insert(0, "row_id", all_row_ids)

# Align to sample submission column order
sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[sample_cols]

out_path = "/kaggle/working/submission.csv"
sub_df.to_csv(out_path, index=False)
print(f"✅ Submission saved: {out_path}")
print(f"   Shape : {sub_df.shape}")
print(sub_df.head(3))